In [2]:
import re
import pandas as pd

# CONFIG — update this path to your file location

RAW_FILE_PATH = "C:/Users/HP/Desktop/Internship Final Submission/NLP/2_raw_jobs.csv"

# DAY 4 — DATA CLEANING

def clean_text(text):
    text = str(text)
    text = re.sub(r"<[^>]+>", " ", text)                    # HTML tags
    text = re.sub(r"http\S+|www\.\S+", " ", text)           # URLs
    text = re.sub(r"\S+@\S+\.\S+", " ", text)               # emails
    text = re.sub(
        r"(\+?\d{1,3}[-.\s]?)?(\(?\d{2,4}\)?[-.\s]?)?\d{3,4}[-.\s]\d{3,4}\b",
        " ", text
    )                                                        # phone numbers
    text = re.sub(r"[^a-zA-Z0-9\s.,!?'-]", " ", text)       # special chars/symbols
    text = re.sub(r"\s+", " ", text)                        # extra spaces
    return text.strip()


def remove_duplicate_sentences(text):
    """Remove repeated sentences within a single description."""
    sentences = re.split(r'(?<=[.!?])\s+', text)
    seen, unique = set(), []
    for s in sentences:
        key = s.strip().lower()
        if key and key not in seen:
            seen.add(key)
            unique.append(s.strip())
    return ' '.join(unique)


def run_day4(raw_path):
    df = pd.read_csv(raw_path)
    print(f"Loaded {len(df)} rows.")

    df["clean_description"] = df["job_description"].apply(clean_text)
    df["clean_description"] = df["clean_description"].apply(remove_duplicate_sentences)

    before = len(df)
    df = df.drop_duplicates(subset=["job_title", "company", "clean_description"], keep="first")
    print(f"Removed {before - len(df)} duplicate postings -> {len(df)} rows remain.")

    df.to_csv("clean_jobs.csv", index=False)
    print("Saved clean_jobs.csv\n")
    return df

# DAY 5 — SKILL TAXONOMY

TAXONOMY = [
    # skill, category, aliases (comma-separated string)
    ("Python", "Programming", "python3, py, python 3"),
    ("Java", "Programming", "java8, java 8, jdk"),
    ("C++", "Programming", "cpp, c plus plus"),
    ("C#", "Programming", "csharp, c sharp"),
    ("C", "Programming", "c language, c programming"),
    ("JavaScript", "Programming", "js, javascript es6, ecmascript"),
    ("TypeScript", "Programming", "ts"),
    ("R", "Programming", "r language, r programming"),
    ("Scala", "Programming", ""),
    ("Go", "Programming", "golang"),
    ("Rust", "Programming", ""),
    ("PHP", "Programming", ""),
    ("Ruby", "Programming", ""),
    ("Swift", "Programming", ""),
    ("Kotlin", "Programming", ""),
    ("MATLAB", "Programming", "matlab r"),
    ("Shell Scripting", "Programming", "bash, shell script, bash scripting"),
    ("VBA", "Programming", "visual basic for applications"),

    ("SQL", "Database", "sql, structured query language"),
    ("MySQL", "Database", "my sql"),
    ("PostgreSQL", "Database", "postgres, postgresql, postgre sql"),
    ("MongoDB", "Database", "mongo, mongo db"),
    ("Oracle DB", "Database", "oracle database, oracle sql"),
    ("SQLite", "Database", "sqlite3"),
    ("Microsoft SQL Server", "Database", "mssql, ms sql server, sql server, t-sql, tsql"),
    ("Cassandra", "Database", "apache cassandra"),
    ("Redis", "Database", ""),
    ("DynamoDB", "Database", "dynamo db, aws dynamodb"),
    ("Snowflake", "Database", ""),
    ("BigQuery", "Database", "google bigquery, gbq"),
    ("Redshift", "Database", "aws redshift"),
    ("Neo4j", "Database", "graph db, neo4j graph database"),
    ("Elasticsearch", "Database", "elastic search, es"),

    ("Power BI", "Data Analytics", "powerbi, power-bi, ms power bi, microsoft power bi"),
    ("Tableau", "Data Analytics", "tableau desktop, tableau server"),
    ("Excel", "Data Analytics", "ms excel, microsoft excel, msexcel, excel vba"),
    ("Google Sheets", "Data Analytics", "gsheets, google spreadsheet"),
    ("Looker", "Data Analytics", "looker studio, google data studio"),
    ("QlikView", "Data Analytics", "qlik, qlik view, qlik sense"),
    ("SAS", "Data Analytics", "sas programming, sas base"),
    ("SPSS", "Data Analytics", "ibm spss"),
    ("Alteryx", "Data Analytics", ""),
    ("Data Visualization", "Data Analytics", "data viz, dataviz"),
    ("Data Analysis", "Data Analytics", "data analytics"),

    ("Machine Learning", "Machine Learning", "ml, machine-learning"),
    ("Deep Learning", "Machine Learning", "dl, deep-learning"),
    ("Scikit-learn", "Machine Learning", "sklearn, scikit learn, scikit-learn"),
    ("TensorFlow", "Machine Learning", "tensor flow, tf, tensorflow2"),
    ("PyTorch", "Machine Learning", "py torch, torch"),
    ("Keras", "Machine Learning", ""),
    ("XGBoost", "Machine Learning", "xg boost, extreme gradient boosting"),
    ("LightGBM", "Machine Learning", "light gbm"),
    ("NLP", "Machine Learning", "natural language processing"),
    ("Computer Vision", "Machine Learning", "cv, opencv, open cv"),
    ("Generative AI", "Machine Learning", "genai, gen ai, generative artificial intelligence"),
    ("LLM", "Machine Learning", "large language models, llms"),
    ("Neural Networks", "Machine Learning", "neural network, ann, artificial neural networks"),
    ("Reinforcement Learning", "Machine Learning", "rl"),
    ("MLOps", "Machine Learning", "ml ops, ml operations"),

    ("Pandas", "Data Engineering", "pandas library, pd"),
    ("NumPy", "Data Engineering", "numpy library, np"),
    ("Spark", "Data Engineering", "apache spark, pyspark, spark sql"),
    ("Hadoop", "Data Engineering", "apache hadoop, hdfs"),
    ("Kafka", "Data Engineering", "apache kafka"),
    ("Airflow", "Data Engineering", "apache airflow"),
    ("ETL", "Data Engineering", "etl pipelines, extract transform load"),
    ("Data Warehousing", "Data Engineering", "data warehouse, dwh"),
    ("Databricks", "Data Engineering", ""),
    ("Hive", "Data Engineering", "apache hive"),
    ("NiFi", "Data Engineering", "apache nifi"),

    ("AWS", "Cloud", "amazon web services, amazon aws"),
    ("Azure", "Cloud", "microsoft azure, ms azure"),
    ("GCP", "Cloud", "google cloud platform, google cloud"),
    ("IBM Cloud", "Cloud", "ibm cloud platform"),
    ("Oracle Cloud", "Cloud", "oci, oracle cloud infrastructure"),

    ("Docker", "DevOps", "docker container, containerization"),
    ("Kubernetes", "DevOps", "k8s, kube"),
    ("Git", "DevOps", "git version control"),
    ("GitHub", "DevOps", "git hub"),
    ("GitLab", "DevOps", "git lab"),
    ("Jenkins", "DevOps", ""),
    ("CI/CD", "DevOps", "ci cd, continuous integration continuous deployment"),
    ("Terraform", "DevOps", ""),
    ("Ansible", "DevOps", ""),
    ("Linux", "DevOps", "unix, linux administration"),
    ("Jira", "DevOps", "atlassian jira"),

    ("HTML", "Web Development", "html5"),
    ("CSS", "Web Development", "css3"),
    ("React", "Web Development", "reactjs, react.js"),
    ("Angular", "Web Development", "angularjs, angular.js"),
    ("Vue.js", "Web Development", "vuejs, vue"),
    ("Node.js", "Web Development", "nodejs, node"),
    ("Django", "Web Development", ""),
    ("Flask", "Web Development", ""),
    ("REST API", "Web Development", "restful api, rest apis"),
    ("Spring Boot", "Web Development", "springboot, spring-boot"),

    ("Figma", "Design", ""),
    ("Adobe Creative Suite", "Design", "adobe, adobe cc, adobe creative cloud"),

    ("Google Analytics", "Marketing", "ga, google analytics 4, ga4"),
    ("SEO", "Marketing", "search engine optimization"),
    ("HubSpot", "Marketing", "hub spot"),

    ("Communication", "Soft Skills", "communication skills, verbal communication"),
    ("Leadership", "Soft Skills", "team leadership, leading teams"),
    ("Problem Solving", "Soft Skills", "problem-solving, analytical thinking"),
    ("Teamwork", "Soft Skills", "team player, collaboration"),
    ("Project Management", "Soft Skills", "project mgmt, pm"),
    ("Agile", "Soft Skills", "agile methodology, scrum"),
    ("Time Management", "Soft Skills", ""),
    ("Critical Thinking", "Soft Skills", ""),
    ("Stakeholder Management", "Soft Skills", "stakeholder mgmt"),
]


def build_taxonomy_csv():
    tax_df = pd.DataFrame(TAXONOMY, columns=["skill", "category", "aliases"])
    tax_df.to_csv("skill_taxonomy.csv", index=False)
    print(f"Saved skill_taxonomy.csv with {len(tax_df)} skills.\n")
    return tax_df


def build_skill_matcher(tax_df):
    """Returns (extract_skills_fn, skill_to_category dict)."""
    alias_to_skill = {}
    skill_to_category = {}

    for _, row in tax_df.iterrows():
        canonical = row["skill"]
        skill_to_category[canonical] = row["category"]
        alias_to_skill[canonical.lower()] = canonical
        if pd.notna(row["aliases"]) and str(row["aliases"]).strip():
            for alias in str(row["aliases"]).split(","):
                alias = alias.strip().lower()
                if alias:
                    alias_to_skill[alias] = canonical

    sorted_aliases = sorted(alias_to_skill.keys(), key=len, reverse=True)
    compiled = {
        a: re.compile(r'(?<!\w)' + re.escape(a) + r'(?!\w)', re.IGNORECASE)
        for a in sorted_aliases
    }

    def extract_skills(text):
        if not isinstance(text, str) or not text.strip():
            return []
        found = set()
        for alias in sorted_aliases:
            if compiled[alias].search(text):
                found.add(alias_to_skill[alias])
        return sorted(found)

    return extract_skills, skill_to_category


def run_day5(df, text_col="clean_description"):
    tax_df = build_taxonomy_csv()
    extract_skills, skill_to_category = build_skill_matcher(tax_df)

    df["extracted_skills"] = df[text_col].apply(extract_skills)
    df["skill_categories"] = df["extracted_skills"].apply(
        lambda skills: sorted(set(skill_to_category[s] for s in skills))
    )
    df["num_skills"] = df["extracted_skills"].apply(len)
    df["extracted_skills_str"] = df["extracted_skills"].apply(lambda x: ", ".join(x))
    df["skill_categories_str"] = df["skill_categories"].apply(lambda x: ", ".join(x))

    output = df.drop(columns=["extracted_skills", "skill_categories"])
    output.to_csv("jobs_with_skills.csv", index=False)
    print("Saved jobs_with_skills.csv")

    print(f"Avg skills/posting: {df['num_skills'].mean():.2f}")
    print(f"Postings with 0 skills matched: {(df['num_skills'] == 0).sum()}\n")

    all_skills = [s for row in df["extracted_skills"] for s in row]
    freq = pd.Series(all_skills).value_counts()
    freq.to_csv("skill_frequency.csv", header=["count"])
    print("Top 15 most in-demand skills:")
    print(freq.head(15))

    return df, freq

# RUN EVERYTHING
if __name__ == "__main__":
    cleaned_df = run_day4(RAW_FILE_PATH)
    final_df, skill_freq = run_day5(cleaned_df)

Loaded 1000 rows.
Removed 0 duplicate postings -> 1000 rows remain.
Saved clean_jobs.csv

Saved skill_taxonomy.csv with 110 skills.

Saved jobs_with_skills.csv
Avg skills/posting: 3.26
Postings with 0 skills matched: 148

Top 15 most in-demand skills:
SQL                       193
Power BI                  155
Excel                     150
CI/CD                     147
Kubernetes                147
Figma                     140
Agile                     137
Docker                    117
AWS                       109
Stakeholder Management    105
Data Analysis             101
Python                     97
Git                        85
Java                       76
Spring Boot                76
Name: count, dtype: int64
